# eCAADe Workshop 2026: Explainable Urban-Tree Growth ML

This exercise fits **one pooled XGBoost model**. Species and monitoring period are categorical predictors; no separate species, conifer, or broadleaf models are trained.

By the end you will be able to:

1. prepare repeated tree observations without leakage;
2. use training, validation, and locked test sets correctly;
3. read SHAP beeswarm, dependence, waterfall, and spatial SHAP plots;
4. export a tested ONNX model with percentage and kg C outputs.

The model target is y = ln(g), where g = [ln(C_end) - ln(C_start)] / years.

## 1. Environment setting and data preparation

Run this cell in a fresh Colab runtime. It installs the libraries used by the exercise.

In [ ]:
# Colab already provides NumPy, pandas, scikit-learn, and Matplotlib.
%pip install -q "xgboost>=2.0,<4" "shap>=0.44,<1" "joblib>=1.3,<2" "onnx>=1.16,<2" "onnxmltools>=1.12,<2" "onnxruntime>=1.17,<2" 

In [ ]:
from pathlib import Path
import subprocess, sys

REPOSITORY = "https://github.com/sleepyheadzzzzzz/Tree-Point-Cloud-Training-and-Analysing.git"
REPO_DIR = Path("/content/Tree-Point-Cloud-Training-and-Analysing")
LOCAL_CANDIDATE = Path.cwd()

if (LOCAL_CANDIDATE / "urban_tree_ml_workshop_2026.py").exists():
    WORKSHOP_DIR = LOCAL_CANDIDATE
else:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORY, str(REPO_DIR)], check=True)
    WORKSHOP_DIR = REPO_DIR / "eCAADe_workshop_2026_material"

sys.path.insert(0, str(WORKSHOP_DIR))
INPUT_CSV = WORKSHOP_DIR / "data" / "tree_carbon_ml_teaching_sample.csv"
OUTPUT_DIR = Path("/content/eCAADe_2026_outputs")
print("Workshop folder:", WORKSHOP_DIR)
print("Teaching CSV:", INPUT_CSV)

In [ ]:
import pandas as pd
from urban_tree_ml_workshop_2026 import (
    WorkflowConfig,
    environment_report,
    load_tree_level_data,
    prepare_data,
    train_validate_refit_test,
    explain_with_shap,
    export_onnx_and_examples,
)

display(pd.Series(environment_report(), name="version").to_frame())
tree_level = load_tree_level_data(INPUT_CSV)
print(f"Tree-level sample: {len(tree_level):,} rows x {tree_level.shape[1]} columns")
display(tree_level.head(3))
display(tree_level.groupby(["Species", "Species_Name"]).size().rename("trees").to_frame())

## 2. Data processing

The tree identifier is split **before** the table is converted to period records. This keeps every observation from one tree in one partition. The function then computes annualized specific growth, log-transforms it, creates period-aligned predictors, and removes invalid/non-positive response rows.

**Exercise:** Check the audit table. Which period loses the most rows, and why might a log-growth target require positive growth?

In [ ]:
config = WorkflowConfig(
    input_csv=INPUT_CSV,
    output_dir=OUTPUT_DIR,
    random_state=2026,
    shap_sample=1200,
    example_rows=12,
)

prepared = prepare_data(config)
display(prepared["processing_audit"])
display(prepared["split_summary"])
display(prepared["long"].head(5))

In [ ]:
# Leakage check: every tree must occur in exactly one partition.
split_counts_per_tree = prepared["long"].groupby("Original_Tree_RowID")["Split"].nunique()
assert split_counts_per_tree.max() == 1
print("Leakage check passed. Maximum partitions per tree:", split_counts_per_tree.max())

display(
    prepared["long"]
    .groupby("Split")[["Log_Annualized_Specific_Growth", "Observed_Annual_Growth_Percent", "Observed_Annual_Carbon_Gain_kg"]]
    .describe()
)

## 3. Model training, validation, and locked test

An initial XGBoost model is fitted to the 70% training partition. Early stopping on the 15% validation partition selects the number of boosting rounds. The preprocessor and model are then refitted on the combined 85% development data, and the locked 15% test is evaluated once.

**Exercise:** Why would repeatedly changing model settings after reading the test score make that score optimistic?

In [ ]:
trained = train_validate_refit_test(prepared, config)
display(trained["metrics"].style.format(precision=4))
print("Selected XGBoost trees:", trained["selected_trees"])
print("Engineered feature count:", trained["x_test"].shape[1])

In [ ]:
import matplotlib.pyplot as plt
pred = pd.read_csv(OUTPUT_DIR / "tables" / "locked_test_predictions.csv")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(pred["Observed_Annual_Growth_Percent"], pred["Predicted_Annual_Growth_Percent"], s=8, alpha=0.25)
limit = pred[["Observed_Annual_Growth_Percent", "Predicted_Annual_Growth_Percent"]].quantile(0.995).max()
ax.plot([0, limit], [0, limit], color="black", linestyle="--", linewidth=1)
ax.set(xlim=(0, limit), ylim=(0, limit), xlabel="Observed annual growth (%)", ylabel="Predicted annual growth (%)", title="Locked test: observed vs predicted")
ax.set_aspect("equal", adjustable="box")
plt.show()

## 4. SHAP explanation

All four plots explain the same pooled XGBoost model on a sample of locked-test observations.

- **Beeswarm:** global distribution of feature contributions.
- **Dependence:** feature value versus SHAP contribution; colour shows the strongest approximate interaction.
- **Waterfall:** one representative prediction.
- **Spatial SHAP:** point map of the leading environmental feature's local contribution.

SHAP values describe fitted associations. They are not causal effects.

In [ ]:
shap_outputs = explain_with_shap(trained, config)
display(shap_outputs["summary"].head(15))
print("Top environmental features used for dependence plots:", shap_outputs["top_environment"])

In [ ]:
from IPython.display import Image, display

for key in ["beeswarm", "dependence", "waterfall", "spatial"]:
    print(key.replace("_", " ").title())
    display(Image(filename=str(shap_outputs[key])))

### Interpretation exercise

1. In the beeswarm, which predictors have the widest contribution distributions?
2. Does a dependence curve look monotonic, threshold-like, or nonlinear?
3. In the waterfall, which features move this observation above or below the expected prediction?
4. Does the spatial SHAP map reveal clusters? What unmeasured local processes could also produce them?
5. Why is the spatial SHAP map not an interpolated suitability map?

## 5. ONNX export and example test set

The ONNX graph accepts the engineered feature matrix and initial carbon stock. It returns raw log-SGR, specific growth rate, annual growth percentage, and annual kg C gain. The export is accepted only after ONNX Runtime reproduces Python XGBoost predictions within tight numerical tolerances.

In [ ]:
onnx_outputs = export_onnx_and_examples(trained, config)
display(pd.Series(onnx_outputs["parity"], name="maximum absolute error").to_frame())
print("ONNX model:", onnx_outputs["onnx_path"])
print("Inputs:", onnx_outputs["schema"]["onnx_inputs"])
print("Outputs:", onnx_outputs["schema"]["onnx_outputs"])

display(pd.read_csv(OUTPUT_DIR / "examples" / "example_test_set_raw.csv").head())
display(pd.read_csv(OUTPUT_DIR / "examples" / "example_onnx_predictions.csv").head())

In [ ]:
# Package every generated table, figure, model, and example.
import shutil
archive = shutil.make_archive("/content/eCAADe_2026_outputs", "zip", OUTPUT_DIR)
print("Created:", archive)

try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Download is available when this notebook runs in Google Colab.")

## Final reminder

This random grouped test is suitable for learning the workflow, but a planning-ready spatial diagnostic needs a spatially blocked deployment test, a reliability/domain mask, and local field verification. Do not attribute the model's full R² to environmental predictors.